# Annolid Training & Inference on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/healthonrails/annolid/blob/main/docs/tutorials/Annolid_Colab_Train_and_Inference.ipynb)

Use this unified notebook to **train** or **run inference** with modern Annolid models (`yolo`, `yolo_labelme`, `dino_kpseg`, `maskrcnn_detectron2`, `keypoint_rcnn`) on Google Colab GPUs.

Tip: You can also launch jobs directly from Annolid GUI (**AI & Models -> Google Colab…**) or CLI (`annolid-run colab train ...` / `annolid-run colab predict ...`).

## 1. Verify Colab GPU & Environment

In [ ]:
import platform, shutil, subprocess
print('Python:', platform.python_version())
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('No GPU attached. Switch Runtime -> Change runtime type -> GPU.')

## 2. Configure Mode, Model, and Drive Paths

In [ ]:
MODE = 'train'  # 'train' or 'predict'
MODEL = 'yolo'  # train: yolo, dino_kpseg, maskrcnn_detectron2, keypoint_rcnn | predict: yolo_labelme, yolo, dino_kpseg, keypoint_rcnn
ANNOLID_EXTRAS = 'yolo'  # 'yolo' for YOLO/YOLOE, or 'ml,training' for DINO KPSEG / Mask R-CNN / Keypoint R-CNN
DRIVE_JOB_DIR = '/content/drive/MyDrive/Annolid/colab/manual_job'
INPUT_ZIP_NAME = 'input_dataset.zip'  # or 'input_inference.zip' for predict
WEIGHTS = 'yolo11s-seg.pt'
EPOCHS = 100
BATCH = 16
IMGSZ = 640
CONF = 0.25

## 3. Install Annolid & Model Extras

In [ ]:
import subprocess, sys
pkg = f'annolid[{ANNOLID_EXTRAS}]' if ANNOLID_EXTRAS else 'annolid'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg])
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 4. Mount Google Drive & Unpack Dataset or Video

In [ ]:
import shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
JOB_ROOT = Path('/content/annolid_job')
INPUT_DIR = JOB_ROOT / 'input'
OUTPUT_DIR = JOB_ROOT / 'output'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

drive_dir = Path(DRIVE_JOB_DIR)
drive_dir.mkdir(parents=True, exist_ok=True)
archive = drive_dir / INPUT_ZIP_NAME
if not archive.is_file():
    from google.colab import files
    print(f'{archive} not found in Drive. Upload your zip archive below:')
    uploaded = files.upload()
    archive = Path('/content') / next(iter(uploaded.keys()))

with zipfile.ZipFile(archive, 'r') as zf:
    for info in zf.infolist():
        dest = (INPUT_DIR / info.filename).resolve()
        if INPUT_DIR.resolve() != dest and INPUT_DIR.resolve() not in dest.parents:
            raise ValueError(f'Unsafe archive path: {info.filename}')
        if info.is_dir():
            dest.mkdir(parents=True, exist_ok=True)
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(info, 'r') as s, open(dest, 'wb') as d:
            shutil.copyfileobj(s, d)
print('Inputs extracted to:', INPUT_DIR)

## 5. Run `annolid-run` Training or Inference

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

def find_first(root: Path, names: tuple[str, ...]) -> Path:
    for name in names:
        matches = sorted(root.rglob(name))
        if matches:
            return matches[0]
    return root

cli = ['annolid-run'] if shutil.which('annolid-run') else [sys.executable, '-m', 'annolid.engine.cli']
if MODE == 'train':
    data_target = find_first(INPUT_DIR, ('data.yaml', 'labelme_spec.yaml', 'coco_spec.yaml'))
    if MODEL == 'yolo':
        cmd = [*cli, 'train', 'yolo', '--data', str(data_target), '--weights', WEIGHTS, '--epochs', str(EPOCHS), '--batch', str(BATCH), '--imgsz', str(IMGSZ), '--project', str(OUTPUT_DIR), '--name', 'run', '--plots']
    elif MODEL in ('maskrcnn_detectron2', 'keypoint_rcnn'):
        cmd = [*cli, 'train', MODEL, '--dataset-dir', str(data_target), '--max-iterations', str(EPOCHS), '--batch-size', str(BATCH), '--output-dir', str(OUTPUT_DIR)]
    else:
        cmd = [*cli, 'train', MODEL, '--data', str(data_target), '--epochs', str(EPOCHS), '--batch', str(BATCH), '--output', str(OUTPUT_DIR)]
else:
    media = next((p for p in sorted(INPUT_DIR.rglob('*')) if p.is_file() and p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv', '.png', '.jpg'}), INPUT_DIR)
    local_weights = next((p for p in sorted((INPUT_DIR / 'weights').glob('*')) if p.is_file()), None) if (INPUT_DIR / 'weights').is_dir() else None
    weight_arg = str(local_weights) if local_weights else WEIGHTS
    if MODEL == 'yolo_labelme':
        cmd = [*cli, 'predict', 'yolo_labelme', '--weights', weight_arg, '--source', str(media), '--output-dir', str(OUTPUT_DIR / 'predictions'), '--conf', str(CONF), '--save-csv']
    else:
        cmd = [*cli, 'predict', MODEL, '--weights', weight_arg, '--source', str(media)]

print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)

## 6. Package Outputs & Save to Google Drive

In [ ]:
import json, shutil, zipfile
from pathlib import Path

bundle = JOB_ROOT / 'output_bundle.zip'
artifacts = []
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUTPUT_DIR.rglob('*')):
        if p.is_file():
            rel = p.relative_to(OUTPUT_DIR).as_posix()
            zf.write(p, rel)
            artifacts.append(rel)

manifest = JOB_ROOT / 'job_result.json'
manifest.write_text(json.dumps({'mode': MODE, 'model': MODEL, 'status': 'completed', 'artifact_count': len(artifacts), 'artifacts': artifacts}, indent=2), encoding='utf-8')
shutil.copy2(bundle, drive_dir / bundle.name)
shutil.copy2(manifest, drive_dir / manifest.name)
print(f'Saved {len(artifacts)} artifact(s) to {drive_dir / bundle.name}')